# Task 2: Restaurant Recommendation System

This task focuses on developing a content-based restaurant recommendation system using the restaurant dataset (`Dataset.csv`). The system recommends restaurants based on user preferences such as cuisine, price range, and city.

## 1. Data Loading and Exploration

In [35]:
import pandas as pd

df = pd.read_csv("Dataset.csv")

print("Dataset Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Dataset Shape: (9551, 21)

Columns:
['Restaurant ID', 'Restaurant Name', 'Country Code', 'City', 'Address', 'Locality', 'Locality Verbose', 'Longitude', 'Latitude', 'Cuisines', 'Average Cost for two', 'Currency', 'Has Table booking', 'Has Online delivery', 'Is delivering now', 'Switch to order menu', 'Price range', 'Aggregate rating', 'Rating color', 'Rating text', 'Votes']


## 2. Data Preprocessing

In [2]:
print("Cuisines:")
print(df["Cuisines"].dropna().head(10))

print("\nPrice Range:")
print(df["Price range"].value_counts().sort_index())

print("\nCities:")
print(df["City"].nunique(), "unique cities")

print("\nOnline Delivery:")
print(df["Has Online delivery"].value_counts())

print("\nTable Booking:")
print(df["Has Table booking"].value_counts())

Cuisines:
0            French, Japanese, Desserts
1                              Japanese
2      Seafood, Asian, Filipino, Indian
3                       Japanese, Sushi
4                      Japanese, Korean
5                               Chinese
6                       Asian, European
7    Seafood, Filipino, Asian, European
8               European, Asian, Indian
9                              Filipino
Name: Cuisines, dtype: object

Price Range:
Price range
1    4444
2    3113
3    1408
4     586
Name: count, dtype: int64

Cities:
141 unique cities

Online Delivery:
Has Online delivery
No     7100
Yes    2451
Name: count, dtype: int64

Table Booking:
Has Table booking
No     8393
Yes    1158
Name: count, dtype: int64


In [3]:
selected_columns = [
    "Cuisines",
    "Price range",
    "City",
    "Has Online delivery",
    "Has Table booking"
]

print(df[selected_columns].isnull().sum())

Cuisines               9
Price range            0
City                   0
Has Online delivery    0
Has Table booking      0
dtype: int64


In [4]:
df["Cuisines"] = df["Cuisines"].fillna("Unknown")

print("Missing Cuisines:", df["Cuisines"].isnull().sum())
print("Unknown Cuisines:", (df["Cuisines"] == "Unknown").sum())

Missing Cuisines: 0
Unknown Cuisines: 9


### 2.1 Feature Selection

In [5]:
recommendation_columns = [
    "Restaurant Name",
    "Cuisines",
    "Price range",
    "City",
    "Has Online delivery",
    "Has Table booking"
]

recommendation_df = df[recommendation_columns].copy()

print("Recommendation dataset shape:", recommendation_df.shape)
print("\nSelected columns:")
print(recommendation_df.columns.tolist())

Recommendation dataset shape: (9551, 6)

Selected columns:
['Restaurant Name', 'Cuisines', 'Price range', 'City', 'Has Online delivery', 'Has Table booking']


In [6]:
matching_features = [
    "Cuisines",
    "Price range",
    "City",
    "Has Online delivery",
    "Has Table booking"
]

X_recommendation = recommendation_df[matching_features].copy()

print("Matching features:")
print(X_recommendation.head())

Matching features:
                           Cuisines  Price range              City  \
0        French, Japanese, Desserts            3       Makati City   
1                          Japanese            3       Makati City   
2  Seafood, Asian, Filipino, Indian            4  Mandaluyong City   
3                   Japanese, Sushi            4  Mandaluyong City   
4                  Japanese, Korean            4  Mandaluyong City   

  Has Online delivery Has Table booking  
0                  No               Yes  
1                  No               Yes  
2                  No               Yes  
3                  No                No  
4                  No               Yes  


### 2.2 Cuisine Data Preparation

In [7]:
# Split the cuisine column into individual cuisine names
X_recommendation["Cuisines"] = X_recommendation["Cuisines"].str.split(",")

print(X_recommendation["Cuisines"].head(10))

0             [French,  Japanese,  Desserts]
1                                 [Japanese]
2      [Seafood,  Asian,  Filipino,  Indian]
3                         [Japanese,  Sushi]
4                        [Japanese,  Korean]
5                                  [Chinese]
6                         [Asian,  European]
7    [Seafood,  Filipino,  Asian,  European]
8                [European,  Asian,  Indian]
9                                 [Filipino]
Name: Cuisines, dtype: object


In [8]:
# Remove extra spaces from each cuisine name
X_recommendation["Cuisines"] = X_recommendation["Cuisines"].apply(
    lambda cuisines: [cuisine.strip() for cuisine in cuisines]
)

print(X_recommendation["Cuisines"].head(10))

0            [French, Japanese, Desserts]
1                              [Japanese]
2      [Seafood, Asian, Filipino, Indian]
3                       [Japanese, Sushi]
4                      [Japanese, Korean]
5                               [Chinese]
6                       [Asian, European]
7    [Seafood, Filipino, Asian, European]
8               [European, Asian, Indian]
9                              [Filipino]
Name: Cuisines, dtype: object


### 2.3 Cuisine Encoding

In [13]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()

cuisine_encoded = mlb.fit_transform(X_recommendation["Cuisines"])

print("Cuisine feature shape:", cuisine_encoded.shape)
print("\nSample cuisine features:")
print(cuisine_encoded[:5])

Cuisine feature shape: (9551, 146)

Sample cuisine features:
[[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0]
 [0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 

### 2.4 City Encoding

In [9]:
from sklearn.preprocessing import OneHotEncoder

city_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

city_encoded = city_encoder.fit_transform(
    X_recommendation[["City"]]
)

print("City feature shape:", city_encoded.shape)
print("Number of cities encoded:", city_encoded.shape[1])

City feature shape: (9551, 141)
Number of cities encoded: 141


### 2.5 Binary Feature Inspection

In [10]:
print("Online Delivery values:")
print(X_recommendation["Has Online delivery"].unique())

print("\nTable Booking values:")
print(X_recommendation["Has Table booking"].unique())

Online Delivery values:
['No' 'Yes']

Table Booking values:
['Yes' 'No']


### 2.6 Binary Feature Encoding

In [11]:
# Convert Yes/No values into 1/0
X_recommendation["Has Online delivery"] = (
    X_recommendation["Has Online delivery"].map({"Yes": 1, "No": 0})
)

X_recommendation["Has Table booking"] = (
    X_recommendation["Has Table booking"].map({"Yes": 1, "No": 0})
)

print(X_recommendation[[
    "Has Online delivery",
    "Has Table booking"
]].head(10))

   Has Online delivery  Has Table booking
0                    0                  1
1                    0                  1
2                    0                  1
3                    0                  0
4                    0                  1
5                    0                  0
6                    0                  1
7                    0                  1
8                    0                  1
9                    0                  1


## 3. Restaurant Feature Representation

In [14]:
import numpy as np

price_encoded = X_recommendation["Price range"].to_numpy().reshape(-1, 1)

delivery_encoded = X_recommendation[
    "Has Online delivery"
].to_numpy().reshape(-1, 1)

booking_encoded = X_recommendation[
    "Has Table booking"
].to_numpy().reshape(-1, 1)

restaurant_features = np.hstack([
    cuisine_encoded,
    city_encoded,
    price_encoded,
    delivery_encoded,
    booking_encoded
])

print("Final feature shape:", restaurant_features.shape)

Final feature shape: (9551, 290)


## 4. User Preference

In [15]:
user_cuisine = "Japanese"
user_price_range = 3
user_city = "Makati City"
user_online_delivery = 0
user_table_booking = 1

print("User Preferences:")
print("Cuisine:", user_cuisine)
print("Price Range:", user_price_range)
print("City:", user_city)
print("Online Delivery:", user_online_delivery)
print("Table Booking:", user_table_booking)

User Preferences:
Cuisine: Japanese
Price Range: 3
City: Makati City
Online Delivery: 0
Table Booking: 1


### 4.1 Cuisine Preference Encoding

In [16]:
user_cuisine_encoded = mlb.transform([[user_cuisine]])

print("User cuisine feature shape:", user_cuisine_encoded.shape)

User cuisine feature shape: (1, 146)


### 4.2 City Preference Encoding

In [17]:
user_city_encoded = city_encoder.transform([[user_city]])

print("User city feature shape:", user_city_encoded.shape)

User city feature shape: (1, 141)


C:\Users\Ashik_T\AppData\Roaming\Python\Python310\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


### 4.3 User Feature Representation

In [19]:
user_price_encoded = np.array([[user_price_range]])

user_features = np.hstack([
    user_cuisine_encoded,
    user_city_encoded,
    user_price_encoded,
    np.array([[user_online_delivery]]),
    np.array([[user_table_booking]])
])

print("User feature shape:", user_features.shape)

User feature shape: (1, 290)


## 5. Content-Based Filtering

In [20]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_scores = cosine_similarity(
    user_features,
    restaurant_features
)

print("Similarity score shape:", similarity_scores.shape)

Similarity score shape: (1, 9551)


### 5.1 Similarity Ranking

In [21]:
# Get the similarity score for each restaurant
scores = similarity_scores[0]

# Find the indices of the top 5 restaurants
top_indices = scores.argsort()[-5:][::-1]

print("Top 5 recommended restaurant indices:")
print(top_indices)

print("\nTop 5 similarity scores:")
print(scores[top_indices])

Top 5 recommended restaurant indices:
[   1    0 1188 3809 4335]

Top 5 similarity scores:
[1.         0.9258201  0.91666667 0.91666667 0.91666667]


### 5.2 Initial Recommendations

In [22]:
recommended_restaurants = recommendation_df.iloc[top_indices].copy()

recommended_restaurants["Similarity Score"] = scores[top_indices]

print(recommended_restaurants.to_string(index=False))

                        Restaurant Name                   Cuisines  Price range        City Has Online delivery Has Table booking  Similarity Score
                       Izakaya Kikufuji                   Japanese            3 Makati City                  No               Yes          1.000000
                       Le Petit Souffle French, Japanese, Desserts            3 Makati City                  No               Yes          0.925820
Club Tokyo - Best Western Skycity Hotel                   Japanese            3     Gurgaon                  No               Yes          0.916667
                                 Tamura                   Japanese            3   New Delhi                  No               Yes          0.916667
                                 Tamura                   Japanese            3   New Delhi                  No               Yes          0.916667


## 6. Core Preference Matching

In [23]:
core_matches = recommendation_df[
    (recommendation_df["Cuisines"].apply(lambda x: user_cuisine in x)) &
    (recommendation_df["Price range"] == user_price_range) &
    (recommendation_df["City"] == user_city)
].copy()

print("Core matching restaurants:", len(core_matches))
print(core_matches.to_string(index=False))


Core matching restaurants: 2
 Restaurant Name                   Cuisines  Price range        City Has Online delivery Has Table booking
Le Petit Souffle French, Japanese, Desserts            3 Makati City                  No               Yes
Izakaya Kikufuji                   Japanese            3 Makati City                  No               Yes


## 7. Recommendation Ranking

In [24]:
recommendation_df["Aggregate rating"] = df.loc[
    recommendation_df.index, "Aggregate rating"
]

recommendation_df["Votes"] = df.loc[
    recommendation_df.index, "Votes"
]

print(recommendation_df[
    ["Restaurant Name", "Aggregate rating", "Votes"]
].head())

          Restaurant Name  Aggregate rating  Votes
0        Le Petit Souffle               4.8    314
1        Izakaya Kikufuji               4.5    591
2  Heat - Edsa Shangri-La               4.4    270
3                    Ooma               4.9    365
4             Sambo Kojin               4.8    229


In [25]:
core_matches = recommendation_df[
    (recommendation_df["Cuisines"].apply(lambda x: user_cuisine in x)) &
    (recommendation_df["Price range"] == user_price_range) &
    (recommendation_df["City"] == user_city)
].copy()

core_matches = core_matches.sort_values(
    by=["Aggregate rating", "Votes"],
    ascending=[False, False]
)

print(
    core_matches[
        [
            "Restaurant Name",
            "Cuisines",
            "Price range",
            "City",
            "Aggregate rating",
            "Votes"
        ]
    ].to_string(index=False)
)

 Restaurant Name                   Cuisines  Price range        City  Aggregate rating  Votes
Le Petit Souffle French, Japanese, Desserts            3 Makati City               4.8    314
Izakaya Kikufuji                   Japanese            3 Makati City               4.5    591


## 8. Recommendation Function

In [26]:
def recommend_restaurants(cuisine, price_range, city, online_delivery, table_booking):
    print("User Preferences:")
    print("Cuisine:", cuisine)
    print("Price Range:", price_range)
    print("City:", city)
    print("Online Delivery:", online_delivery)
    print("Table Booking:", table_booking)

    matches = recommendation_df[
        (recommendation_df["Cuisines"].apply(lambda x: cuisine in x)) &
        (recommendation_df["Price range"] == price_range) &
        (recommendation_df["City"] == city)
    ].copy()

    print("\nMatching restaurants:", len(matches))
    matches = matches.sort_values(
        by=["Aggregate rating", "Votes"],
        ascending=[False, False]
    )

    return matches[
        [
            "Restaurant Name",
            "Cuisines",
            "Price range",
            "City",
            "Has Online delivery",
            "Has Table booking",
            "Aggregate rating",
            "Votes"
        ]
    ]

## 9. Testing the Recommendation System

In [27]:
recommendations = recommend_restaurants(
    cuisine="Japanese",
    price_range=3,
    city="Makati City",
    online_delivery=0,
    table_booking=1
)

print("\nRecommended Restaurants:")
print(recommendations.to_string(index=False))

User Preferences:
Cuisine: Japanese
Price Range: 3
City: Makati City
Online Delivery: 0
Table Booking: 1

Matching restaurants: 2

Recommended Restaurants:
 Restaurant Name                   Cuisines  Price range        City Has Online delivery Has Table booking  Aggregate rating  Votes
Le Petit Souffle French, Japanese, Desserts            3 Makati City                  No               Yes               4.8    314
Izakaya Kikufuji                   Japanese            3 Makati City                  No               Yes               4.5    591


### 9.1 Checking Valid Test Values

In [28]:
print(df["Cuisines"].dropna().head(20).tolist())
print("\nCities:")
print(df["City"].drop_duplicates().head(20).tolist())
print("\nPrice ranges:")
print(sorted(df["Price range"].unique()))

['French, Japanese, Desserts', 'Japanese', 'Seafood, Asian, Filipino, Indian', 'Japanese, Sushi', 'Japanese, Korean', 'Chinese', 'Asian, European', 'Seafood, Filipino, Asian, European', 'European, Asian, Indian', 'Filipino', 'Filipino, Mexican', 'American, Ice Cream, Desserts', 'Filipino, Mexican', 'Filipino', 'Korean', 'Cafe, American, Italian, Filipino', 'Italian, Pizza', 'Filipino', 'Cafe, Korean, Desserts', 'Cafe, Bakery, American, Italian']

Cities:
['Makati City', 'Mandaluyong City', 'Pasay City', 'Pasig City', 'Quezon City', 'San Juan City', 'Santa Rosa', 'Tagaytay City', 'Taguig City', 'Bras�_lia', 'Rio de Janeiro', 'S��o Paulo', 'Albany', 'Armidale', 'Athens', 'Augusta', 'Balingup', 'Beechworth', 'Boise', 'Cedar Rapids/Iowa City']

Price ranges:
[1, 2, 3, 4]


### 9.2 Test Case 2 — No Exact Match

In [29]:
test_2 = recommend_restaurants(
    cuisine="Chinese",
    price_range=1,
    city="Makati City",
    online_delivery=0,
    table_booking=0
)

print("\nTest 2 Recommendations:")
print(test_2.to_string(index=False))

User Preferences:
Cuisine: Chinese
Price Range: 1
City: Makati City
Online Delivery: 0
Table Booking: 0

Matching restaurants: 0

Test 2 Recommendations:
Empty DataFrame
Columns: [Restaurant Name, Cuisines, Price range, City, Has Online delivery, Has Table booking, Aggregate rating, Votes]
Index: []


### 9.3 Investigating Chinese Restaurants

In [30]:
chinese_restaurants = recommendation_df[
    recommendation_df["Cuisines"].apply(lambda x: "Chinese" in x)
]

print("Chinese restaurants:", len(chinese_restaurants))

print(
    chinese_restaurants[
        ["Restaurant Name", "Cuisines", "Price range", "City"]
    ].head(20).to_string(index=False)
)

Chinese restaurants: 2733
                  Restaurant Name                                     Cuisines  Price range                   City
                     Din Tai Fung                                      Chinese            3       Mandaluyong City
    Guang Zhou Chinese Restaurant                   Asian, Chinese, Vegetarian            1                 Albany
                   Hong Kong Cafe                 Chinese, Seafood, Vegetarian            2                 Albany
     House of China Restaurant II                                      Chinese            1                 Albany
Miyabi Kyoto Japanese Steak House                               Chinese, Steak            3                Augusta
             Thai Moon Restaurant                         Chinese, Sushi, Thai            1 Cedar Rapids/Iowa City
               Ting's Red Lantern                                      Chinese            1 Cedar Rapids/Iowa City
     Chef Lee's Peking Restaurant                     

### 9.4 Test Case 2 — Valid Preference

In [31]:
test_2 = recommend_restaurants(
    cuisine="Chinese",
    price_range=1,
    city="Albany",
    online_delivery=0,
    table_booking=0
)

print("\nTest 2 Recommendations:")
print(test_2.to_string(index=False))

User Preferences:
Cuisine: Chinese
Price Range: 1
City: Albany
Online Delivery: 0
Table Booking: 0

Matching restaurants: 2

Test 2 Recommendations:
              Restaurant Name                   Cuisines  Price range   City Has Online delivery Has Table booking  Aggregate rating  Votes
Guang Zhou Chinese Restaurant Asian, Chinese, Vegetarian            1 Albany                  No                No               3.9    141
 House of China Restaurant II                    Chinese            1 Albany                  No                No               3.8    153


### 9.5 Test Case 3 — Indian

In [32]:
test_3 = recommend_restaurants(
    cuisine="Indian",
    price_range=2,
    city="New Delhi",
    online_delivery=1,
    table_booking=0
)

print("\nTest 3 Recommendations:")
print(test_3.to_string(index=False))

User Preferences:
Cuisine: Indian
Price Range: 2
City: New Delhi
Online Delivery: 1
Table Booking: 0

Matching restaurants: 948

Test 3 Recommendations:
                      Restaurant Name                                                                                 Cuisines  Price range      City Has Online delivery Has Table booking  Aggregate rating  Votes
                          Food Scouts                                                       North Indian, Chinese, Continental            2 New Delhi                 Yes                No               4.6     61
                  Midnight Hunger Hub                                                  North Indian, Fast Food, Italian, Asian            2 New Delhi                 Yes                No               4.5     50
                        Carnatic Cafe                                                                             South Indian            2 New Delhi                  No                No               4.4   

## 10. Recommendation Validation

In [33]:
print("Cuisine matches:")
print(test_3["Cuisines"].apply(lambda x: "Indian" in x).all())

print("\nPrice range matches:")
print((test_3["Price range"] == 2).all())

print("\nCity matches:")
print((test_3["City"] == "New Delhi").all())

Cuisine matches:
True

Price range matches:
True

City matches:
True


### 10.1 Validation of Test Cases 1 and 2

In [34]:
print("TEST 1")

print("Cuisine matches:",
      recommendations["Cuisines"].apply(lambda x: "Japanese" in x).all())

print("Price range matches:",
      (recommendations["Price range"] == 3).all())

print("City matches:",
      (recommendations["City"] == "Makati City").all())


print("\nTEST 2")

print("Cuisine matches:",
      test_2["Cuisines"].apply(lambda x: "Chinese" in x).all())

print("Price range matches:",
      (test_2["Price range"] == 1).all())

print("City matches:",
      (test_2["City"] == "Albany").all())

TEST 1
Cuisine matches: True
Price range matches: True
City matches: True

TEST 2
Cuisine matches: True
Price range matches: True
City matches: True
